这里我们不使用任何的深度学习框架提供的一些计算，从零开始开始实现，包括数据流水线，模型、损失函数和小批量随机梯度下降优化器

In [74]:
%matplotlib inline
import random
import torch


根据带有噪声的线形模型构造一个人造数据集。我们使用线形模型参数 $w=[2,-3.4]^T$、$b=4.2$ 和噪声项 $\epsilon$ 生成数据集及其标签

$Y = Xw+b+\epsilon$

In [75]:
def synthetic_data(w, b, num_examples):
    """
    生成 y = Xw + b + 噪声
    w: 权重
    b: 偏差
    num_examples: 样本数量
    """
    # 生成一组随机数，服从均值为0，标准差为1的正态分布
    X = torch.normal(0, 1, (num_examples, len(w)))    
    # 计算y的值      
    y = torch.matmul(X, w) + b
    # 给y添加噪声，服从均值为0，标准差为0.01的正态分布
    y += torch.normal(0, 0.01, y.shape)

    return X, y.reshape((-1, 1))

true_w = torch.tensor([2, -3.4])
true_b = 4.2
# 生成数据特征和标签
features, labels = synthetic_data(true_w, true_b, 1000)

features中每一行都包含一个二维数据样本，labels中的每一行都包含一维标签值（一个标量）

In [76]:
print('features:', features[0],'\nlabel:', labels[0])

features: tensor([1.3963, 0.6294]) 
label: tensor([4.8592])


定义一个data_iter函数，该函数接收批量大小、特征矩阵和标签向量作为输入，生成大小为batch_size的小批量

In [77]:
def data_iter(batch_size, features, labels):
    """
    生成一个大小为batch_size的小批量
    batch_size: 批量大小
    features: 特征矩阵
    labels: 标签向量
    """
    num_examples = len(features)             # 获取总样本数
    indices = list(range(num_examples))      # 生成一个从0到n-1的连续整数列表。每个数字代表数据集中的一个样本索引。
    # 随机打乱索引
    random.shuffle(indices)                 
    
    for i in range(0, num_examples, batch_size):
        batch_indices = torch.tensor(
            indices[i: min(i + batch_size, num_examples)]
        )
        yield features[batch_indices], labels[batch_indices]

batch_size = 10
for X, y in data_iter(batch_size, features, labels):
    print(X, '\n', y)
    break

tensor([[-0.1326, -0.4512],
        [-1.5496, -0.0526],
        [-0.5298, -2.3304],
        [ 0.5027,  0.6211],
        [ 1.8881,  0.3965],
        [-0.7332, -0.2092],
        [ 0.6000, -1.1004],
        [-1.4350,  1.6943],
        [-0.9954, -0.1299],
        [ 0.1478, -1.0373]]) 
 tensor([[ 5.4683],
        [ 1.2929],
        [11.0809],
        [ 3.0860],
        [ 6.6144],
        [ 3.4383],
        [ 9.1309],
        [-4.4337],
        [ 2.6569],
        [ 8.0155]])


定义初始化模型参数

In [78]:
w = torch.normal(0, 0.01, size=(2, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

定义线性回归模型

In [79]:
def linear(X, w, b):
    """
    线性回归模型
    x: 特征
    w: 权重
    b: 偏差
    """
    return torch.matmul(X, w) + b

定义损失函数

In [80]:
def squared_loss(y_hat, y):
    """
    均方损失
    y_hat: 预测值
    y: 标签值
    """
    return (y_hat - y.reshape(y_hat.shape)) ** 2 / 2

定义优化算法

In [81]:
def sgd(params, lr, batch_size):
    """
    小批量随机梯度下降
    params: 模型参数
    lr: 学习率
    batch_size: 批量大小
    """
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

训练过程

In [82]:
lr = 0.031
num_epochs = 10             # 训练回合
net = linear                # 模型
loss = squared_loss         # 损失函数

for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        l = loss(net(X, w, b), y)       # 计算损失
        l.sum().backward()              # 反向传播计算梯度
        sgd([w, b], lr, batch_size)     # 优化器更新参数

    # 训练结束后，使用整个数据集计算损失并打印（不计入梯度计算）
    with torch.no_grad():
        train_l = loss(net(features, w, b), labels)
        print(f'epoch {epoch + 1}, loss {float(train_l.mean()):f}')

epoch 1, loss 0.027830
epoch 2, loss 0.000106
epoch 3, loss 0.000050
epoch 4, loss 0.000050
epoch 5, loss 0.000050
epoch 6, loss 0.000050
epoch 7, loss 0.000050
epoch 8, loss 0.000050
epoch 9, loss 0.000050
epoch 10, loss 0.000050


比较真实参数和通过训练学到的参数来评估训练的成功的程度

In [83]:
print(f'w的估计误差: {true_w - w.reshape(true_w.shape)}')
print(f'b的估计误差: {true_b - b}')

w的估计误差: tensor([-0.0002,  0.0005], grad_fn=<SubBackward0>)
b的估计误差: tensor([0.0005], grad_fn=<RsubBackward1>)
